# Geovisor - Hospitales y Clinicas ITT

**Proyecto:** Gobierno de Datos 2026

**Modulo:** Espacializacion de Instituciones Prestadoras de Salud

**Repositorio:** [GitHub](https://github.com/j0rg3c45/Hospitales_Cl-nicas_ITT.git)

---

## Objetivo

Generar un visor geografico interactivo (Geovisor) que permita visualizar, filtrar y analizar individualmente cada hospital y clinica del territorio, consumiendo servicios de imagenes satelitales de Google como mapa base.

---
## 1. Instalacion de Dependencias

In [11]:
%pip install folium --quiet

---
## 2. Importacion de Librerias

In [12]:
import json
import folium
from folium import FeatureGroup, LayerControl
from folium.plugins import MiniMap, Fullscreen, LocateControl

print("Librerias importadas.")
print(f"  Folium: {folium.__version__}")

Librerias importadas.
  Folium: 0.20.0


---
## 3. Datos de Hospitales y Clinicas

In [13]:
# Datos embebidos directamente para garantizar funcionamiento en Colab
features = [
    {
        "nombre": "Hospital Terron Colorado - ESE Ladera",
        "direccion": "Av. 4a Oe.",
        "comuna": "COMUNA 1",
        "municipio": "Cali",
        "departamento": "Valle del Cauca",
        "lat": 3.4524408768046295,
        "lon": -76.55915347465202
    },
    {
        "nombre": "Centro De Salud La Rivera",
        "direccion": "Cra. 1g #65-35",
        "comuna": "Comuna 5",
        "municipio": "Cali",
        "departamento": "Valle del Cauca",
        "lat": 3.474079476955456,
        "lon": -76.49050412813828
    },
    {
        "nombre": "Centro De Salud Antonio Narino",
        "direccion": "Cra. 40 #38-99",
        "comuna": "Comuna 38",
        "municipio": "Cali",
        "departamento": "Valle del Cauca",
        "lat": 3.4162184243856415,
        "lon": -76.5083103585888
    }
]

print(f"Datos cargados: {len(features)} instituciones.")
for i, f in enumerate(features):
    print(f"  {i+1}. {f['nombre']} ({f['lat']:.4f}, {f['lon']:.4f})")

Datos cargados: 3 instituciones.
  1. Hospital Terron Colorado - ESE Ladera (3.4524, -76.5592)
  2. Centro De Salud La Rivera (3.4741, -76.4905)
  3. Centro De Salud Antonio Narino (3.4162, -76.5083)


---
## 4. Espacializacion / Geovisor

Se construye el mapa interactivo con:

- Mapa base: Google Satellite, Google Hybrid, Google Maps, OpenStreetMap.
- Capas individuales: cada hospital/clinica como capa independiente.
- Layer Control: permite encender/apagar cada institucion.
- Popups informativos con datos de cada IPS.

In [14]:
# Tiles de Google
GOOGLE_SATELLITE = "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}"
GOOGLE_MAPS = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}"
GOOGLE_HYBRID = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}"

# Centro del mapa
center_lat = sum(f['lat'] for f in features) / len(features)
center_lon = sum(f['lon'] for f in features) / len(features)

# Crear mapa
mapa = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles=None,
    control_scale=True
)

# Basemaps
folium.TileLayer(tiles=GOOGLE_SATELLITE, attr="Google", name="Google Satellite", overlay=False).add_to(mapa)
folium.TileLayer(tiles=GOOGLE_HYBRID, attr="Google", name="Google Hybrid", overlay=False).add_to(mapa)
folium.TileLayer(tiles=GOOGLE_MAPS, attr="Google", name="Google Maps", overlay=False).add_to(mapa)
folium.TileLayer(tiles="OpenStreetMap", name="OpenStreetMap", overlay=False).add_to(mapa)

# Capas individuales por institucion
colores = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'darkblue', 'darkgreen']

for idx, f in enumerate(features):
    color = colores[idx % len(colores)]
    fg = FeatureGroup(name=f['nombre'], show=True)

    popup_html = (
        f"<div style='font-family:Arial;font-size:12px;min-width:220px;'>"
        f"<h4 style='margin-bottom:5px;'>{f['nombre']}</h4>"
        f"<hr style='margin:3px 0;'>"
        f"<b>Direccion:</b> {f['direccion']}<br>"
        f"<b>Comuna:</b> {f['comuna']}<br>"
        f"<b>Municipio:</b> {f['municipio']}<br>"
        f"<b>Departamento:</b> {f['departamento']}<br>"
        f"<b>Lat:</b> {f['lat']:.6f}<br>"
        f"<b>Lon:</b> {f['lon']:.6f}<br>"
        f"</div>"
    )

    folium.Marker(
        location=[f['lat'], f['lon']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=f['nombre'],
        icon=folium.Icon(color=color, icon='plus-sign', prefix='glyphicon')
    ).add_to(fg)

    fg.add_to(mapa)

# Controles
LayerControl(collapsed=False, position='topright').add_to(mapa)
MiniMap(toggle_display=True, position='bottomleft').add_to(mapa)
Fullscreen(position='topleft', title='Pantalla completa', title_cancel='Salir').add_to(mapa)
LocateControl(strings={'title': 'Mi ubicacion'}).add_to(mapa)

print(f"Geovisor construido con {len(features)} capas. Centro: [{center_lat:.4f}, {center_lon:.4f}]")

Geovisor construido con 3 capas. Centro: [3.4476, -76.5193]


---
## 5. Visualizacion

In [17]:
from IPython.display import display, HTML
display(mapa)

---
## 6. Exportacion